# E00 · Reproduce the source and the environment

**Outcome:** Verify that the learning graph comes from published clinical records and that every transformation is traceable.

**Time:** about 30 minutes. Run cells in order. This is the executed solution edition.

This course uses 60 encounters selected from Kaggle's brandao/diabetes dataset, version 1. The underlying UCI resource describes hospital encounters from 1999–2008. Every selected value is retained in data/encounters.csv. We select 20 rows each for source primary diagnosis tokens 250.02, 428 and 493, ordered by numeric encounter ID. This makes the cohort small enough to inspect and deliberately balanced for learning. It is not a representative hospital sample.

Read the source manifest before querying. Source diagnoses are ICD-9-CM; the later code systems are independent reference layers. Missing and categorical values remain visible. A1Cresult is a category, not a numeric laboratory observation. The source value None records the absence of an HbA1c measurement in the source dictionary; it is not a normal result. A dataset about diabetes cannot support population-wide prevalence estimates.

The core notebooks run offline after installing Python packages and Java 17. RDFLib handles graph storage and SPARQL; pySHACL checks explicit data contracts; OWLAPI checks profiles; HermiT handles OWL DL. Owlready2 supplies the packaged reasoner and JPype connects Python to Java. A pinned BFO core is included so imports resolve locally. The requirements-lock file records the environment used to execute the solutions.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Inspect the manifest and verify the selected CSV

In [2]:
manifest=json.loads((ROOT/'data/source_manifest.json').read_text())
assert digest(ROOT/'data/encounters.csv') == manifest['subset_sha256']
print({k:manifest[k] for k in ['kaggle_ref','kaggle_version','source_rows','subset_rows','source_coding']})

{'kaggle_ref': 'brandao/diabetes', 'kaggle_version': 1, 'source_rows': 101766, 'subset_rows': 60, 'source_coding': 'ICD-9-CM as reported in source; category precision varies. Do not reinterpret as ICD-10-CM.'}


## Read strings before casting only measured counts

In [3]:
import pandas as pd
frame=pd.read_csv(ROOT/'data/encounters.csv',dtype=str,keep_default_na=False)
display(frame.head())
display(frame.groupby('diag_1').size().rename('encounters'))
assert set(frame.diag_1)=={'250.02','428','493'}

,encounter_id,patient_nbr,diag_1,time_in_hospital,num_lab_procedures,A1Cresult,readmitted
0,2595612,89193870,250.02,2,53,>8,>30
1,2865816,8450352,250.02,1,34,None,NO
2,2913624,5073354,250.02,1,36,>8,NO
3,3914808,3678201,250.02,3,70,>8,>30
4,4084524,76959585,250.02,10,72,>8,>30


diag_1
250.02    20
428       20
493       20
Name: encounters, dtype: int64

## Your turn

Write Python that counts the records with a source readmission category of <30. This is a count in this selected dataset, not a risk model.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [4]:
answer = sum(r['readmitted']=='<30' for r in rows())

In [5]:
learner_check(answer, lambda x: x == 8, 'Read the category as a string; preserve the less-than sign.')

Exercise passed.
Out[0]: True


## Explain your model

Explain why neither the cohort selection nor a code mapping establishes a causal claim about readmission.

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** Compare the proof premises, source scope and query contract described above; use your own words in a peer review.